# 01.9 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/09-eda-checklist/exercises.ipynb)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

(541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

541,910 rows → 392,693 after cleaning


In [4]:
def overview(df, target=None):
    """Checklist steps 1-5 in one call. Prints; returns nothing."""
    print(f"1. SHAPE      {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("2. TYPES")
    types = pd.DataFrame({"dtype": df.dtypes.astype(str), "distinct": df.nunique()})
    print(types.to_string())
    miss = df.isna().mean().mul(100).round(1)
    print("3. MISSING %  " + (miss[miss > 0].to_dict().__repr__() if (miss > 0).any() else "none"))
    print(f"4. DUPLICATES {df.duplicated().sum():,} exact")
    if target is not None:
        t = df[target]
        if t.nunique() <= 10:
            print(f"5. TARGET     {t.value_counts(normalize=True).round(3).to_dict()}")
        else:
            print(f"5. TARGET     mean {t.mean():.1f}, median {t.median():.1f}, min {t.min():.1f}, max {t.max():.1f}, zeros {(t == 0).mean():.0%}")

## Exercise 1: EDA on transactions

Run `overview(raw)` on the raw transactions. Write five sentences of findings. Which checklist steps do not apply at the transaction level, and why?

In [5]:
# your code here

In [6]:
cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"),
    first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"),
    country=("Country", "first"))
cust["avg_order"] = cust["spend"] / cust["orders"]
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days

# target: what each customer spent in the next quarter (0 if they did not come back)
cust["spend_next_quarter"] = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["came_back"] = (cust["spend_next_quarter"] > 0).astype(int)

# a column "helpfully" added by the export: recency measured at export time (Dec 2011)
last_ever = sales.groupby("Customer ID")["InvoiceDate"].max()
cust["recency_days"] = (sales["InvoiceDate"].max() - last_ever).dt.days.reindex(cust.index)

cust = cust.drop(columns=["first_order", "last_order"])
print(cust.shape)
cust.head()

(3317, 11)


,orders,spend,products,items,country,avg_order,tenure_days,days_since_last,spend_next_quarter,came_back,recency_days
Customer ID,,,,,,,,,,,
12346,1,77183.60,1,74215,United Kingdom,77183.600000,225,225,0.00,0,325
12347,5,2790.86,82,1590,Iceland,558.172000,267,29,1294.32,1,1
12348,3,1487.24,22,2124,Finland,495.746667,258,148,310.00,1,74
12350,1,334.40,17,197,Norway,334.400000,210,210,0.00,0,309
12352,5,1561.81,26,254,Norway,312.362000,196,162,944.23,1,35


## Exercise 2: the classification target

Imbalance ratio of `came_back`. Then, for each numeric feature, the median per class; which feature separates the classes most (largest ratio of medians)? Exclude `recency_days`.

In [7]:
# your code here

## Exercise 3: an informative missing value

Add `avg_days_between_orders` = (last order − first order) / (orders − 1) in days, computed from `past`. What is its missing rate, why is it missing, and is that informative about `came_back`?

In [8]:
# your code here

## Exercise 4: redundant pairs

List all feature pairs with |correlation| > 0.6 (excluding the targets and `recency_days`). For each pair, say which you would keep and why.

In [9]:
# your code here